In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import norm
import pandas as pd
from scipy import stats
from itertools import product
from scipy import linalg
from functools import partial

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.dgp import GaussianNetwork, BernoulliNetwork
from src.metrics import (
    TrueRejection,
    FalseRejection,
    Rejection,
    RelativeFrobeniusNorm,
    rv_coefficient_adjusted,
    rv_coefficient,
    ComputeAll,
)
from src.methods import (
    RVPermutationTest,
    FitIndependent,
    LLKRatioTest,
    DiffusionCorrelation,
    CanonicalCorrelationTest
)
from src.solvers import MLE_logistic, ASE, MLE_gaussian
from src.simulation_functions import run_simulation
from src.analyse_functions import aggregate_results
from src.plot_functions import plot_grid, plot_with_bands
import scipy.stats as stats
from src.methods import QAP
from src.plot_functions import visualise_latent

In [ ]:
def check_normality(data, label):
    print(f"--- Normality Report for: {label} ---")

    # --- Descriptive Statistics ---
    skew = stats.skew(data)
    kurt = stats.kurtosis(data)
    print(f"Skewness: {skew:.4f} (Ideal: 0)")
    print(f"Excess Kurtosis: {kurt:.4f} (Ideal: 0)")

    # --- Shapiro-Wilk Test ---
    # Good for small to medium samples (n < 5000)
    shapiro_stat, shapiro_p = stats.shapiro(data)
    print(
        f"Shapiro-Wilk Test: p={shapiro_p:.4f} ({'Normal' if shapiro_p > 0.05 else 'Not Normal'})"
    )

    # --- D'Agostino's K^2 Test ---
    k2_stat, k2_p = stats.normaltest(data)
    print(
        f"D'Agostino's K^2: p={k2_p:.4f} ({'Normal' if k2_p > 0.05 else 'Not Normal'})"
    )

    # --- Anderson-Darling Test ---
    result = stats.anderson(data)
    print(f"Anderson-Darling Stat: {result.statistic:.4f}")
    # Compare against critical value at 5% (index 2)
    is_normal = result.statistic < result.critical_values[2]
    print(f"Anderson-Darling (5%): {'Normal' if is_normal else 'Not Normal'}\n")

In [ ]:
# out = []
# for i in range(10):
#     m = BernoulliNetwork(n=500, k=2, sigma=0)
#     data = m.generate()
#     A = data['A']
#     B = data['B']
#     X = data['X']
#     Z = data['Z']

#     rng = np.random.default_rng()

#     Xhat = MLE_logistic(A, k=2, rng=rng)
#     Zhat = MLE_logistic(B, k=2, rng=rng)

#     Xhat = Xhat[0][0]
#     Zhat = Zhat[0][0]

#     # check_normality(Xhat[:, 0], "Xhat")
#     # check_normality(X[:, 0], "X")

#     # fig, ax = plt.subplots(1, 1, figsize=(7, 5))
#     # ax.hist(Xhat[:, 0], alpha=0.75)
#     # ax.hist(X[:, 0], alpha=0.75)
#     # plt.legend(["$X_{hat}$", "$X$"])
#     # plt.show()

#     out.append(norm(Xhat))

# sum(out) / len(out)

In [ ]:
m = GaussianNetwork(n=200, k=5, rho=0, copula_model='gumbel')
data = m.generate()
A = data['A']
B = data['B']
X = data['X']
Z = data['Z']

rng = np.random.default_rng(42)

X_model = FitIndependent(rng=rng, k=2, solver=MLE_logistic)
X_model.fit(data)
Z_model = FitIndependent(rng=rng, k=2, solver=MLE_logistic)
Z_model.fit(data)

Xhat = X_model.get_estimated()['estimated_latent'][0]
Zhat = Z_model.get_estimated()['estimated_latent'][0]

check_normality(Xhat[:, 0], "Xhat")
check_normality(X[:, 0], "X")

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.hist(Xhat[:, 0], alpha=0.75)
ax.hist(X[:, 0], alpha=0.75)
plt.legend(["$X_{hat}$", "$X$"])
plt.show()